# Compression and Momentum Study

## Objective

This notebook investigates whether periods of unusually low volatility are followed by directional price movement.

## Key Concept: Volatility Compression

Volatility compression occurs when price movement becomes unusually small relative to recent history.

Such periods often precede volatility expansion.

## Research Questions

- Does compression lead to breakout behavior?
- Can momentum improve breakout selection?
- Does combining both signals improve performance?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("./NIFTY_50_minute.csv")

df['date'] = pd.to_datetime(
    df['date'],
    format='%d-%m-%Y %H:%M'
)

df = df.sort_values('date')
df = df.set_index('date')

df.head()

,open,high,low,close,volume
date,,,,,
2015-01-09 09:15:00,8285.45,8295.90,8285.45,8292.10,0
2015-01-09 09:16:00,8292.60,8293.60,8287.20,8288.15,0
2015-01-09 09:17:00,8287.40,8293.90,8287.40,8293.90,0
2015-01-09 09:18:00,8294.25,8300.65,8293.90,8300.65,0
2015-01-09 09:19:00,8300.60,8301.30,8298.75,8301.20,0


In [ ]:
# ============================================================
# DATA CLEANING
# ============================================================

market_open = pd.Timestamp("09:15").time()
market_close = pd.Timestamp("15:29").time()

df = df[
    (df.index.time >= market_open) &
    (df.index.time <= market_close)
].copy()

df = df[
    ~df.index.duplicated(keep="first")
]

print("Rows:", len(df))
print("Start:", df.index.min())
print("End:", df.index.max())

## Key Concept: Volatility Compression

Periods of unusually low volatility often precede volatility expansion.

The hypothesis is:

Low Volatility -> Market Energy Builds -> Breakout -> Momentum Follow-Through

This notebook combines:

1. Opening Range Compression
2. Morning Momentum

to determine whether compression improves momentum-based forecasting.

## Research Question

Can volatility compression identify higher-quality momentum signals?

In [ ]:
or_stats = []

for day, day_df in df.groupby(df.index.date):

    opening_range = day_df.between_time(
        "09:15",
        "09:29"
    )

    if len(opening_range) < 10:
        continue

    day_open = opening_range.iloc[0]['open']

    or_high = opening_range['high'].max()
    or_low = opening_range['low'].min()

    or_width = (
        (or_high - or_low)
        / day_open
    )

    or_stats.append({
        "date": pd.Timestamp(day),
        "or_width": or_width
    })

or_df = pd.DataFrame(or_stats)

or_df["median_width"] = (
    or_df["or_width"]
    .rolling(20)
    .median()
    .shift(1)
)

or_df.head()

,date,or_width,median_width
0,2015-01-09,0.002118,NaN
1,2015-01-12,0.005518,NaN
2,2015-01-13,0.003774,NaN
3,2015-01-14,0.003286,NaN
4,2015-01-15,0.008534,NaN


In [ ]:
# ==========================================
# BUILD MORNING MOMENTUM DATASET
# ==========================================

research = []

for day, day_df in df.groupby(df.index.date):

    open_bar = day_df.between_time(
        "09:15",
        "09:15"
    )

    eleven_bar = day_df.between_time(
        "11:00",
        "11:00"
    )

    close_bar = day_df.between_time(
        "15:15",
        "15:15"
    )

    if (
        len(open_bar) == 0
        or len(eleven_bar) == 0
        or len(close_bar) == 0
    ):
        continue

    day_open = open_bar.iloc[0]["open"]

    eleven_close = eleven_bar.iloc[0]["close"]

    day_close = close_bar.iloc[0]["close"]

    morning_return = (
        eleven_close / day_open - 1
    )

    research.append({
        "date": pd.Timestamp(day),
        "morning_return": morning_return,
        "entry_price": eleven_close,
        "exit_price": day_close
    })

research = pd.DataFrame(research)

# ==========================================
# MERGE WITH OR DATA
# ==========================================

research = research.merge(
    or_df,
    on="date",
    how="left"
)

research.head()

,date,morning_return,entry_price,exit_price,or_width,median_width
0,2015-01-09,0.000278,8287.75,8285.45,0.002118,NaN
1,2015-01-12,-0.000476,8287.40,8320.60,0.005518,NaN
2,2015-01-13,-0.000821,8339.30,8303.20,0.003774,NaN
3,2015-01-14,-0.000536,8302.80,8270.20,0.003286,NaN
4,2015-01-15,-0.000771,8418.70,8508.25,0.008534,NaN


In [ ]:
threshold = 0.0075

trade_returns = []
trade_dates = []

for _, row in research.iterrows():

    if pd.isna(row["median_width"]):
        continue

    # Compression Filter

    if row["or_width"] > row["median_width"]:
        continue

    morning_return = row["morning_return"]

    # Long

    if morning_return > threshold:

        ret = (
            row["exit_price"]
            - row["entry_price"]
        ) / row["entry_price"]

        ret -= 0.0005

    # Short

    elif morning_return < -threshold:

        ret = (
            row["entry_price"]
            - row["exit_price"]
        ) / row["entry_price"]

        ret -= 0.0005

    else:
        continue

    trade_returns.append(ret)
    trade_dates.append(row["date"])

In [ ]:
trades = pd.Series(
    trade_returns,
    index=pd.to_datetime(trade_dates)
)

equity = (1 + trades).cumprod()

total_return = equity.iloc[-1] - 1

years = (
    (trades.index[-1]
     - trades.index[0]).days
    / 365.25
)

cagr = (
    equity.iloc[-1]
    ** (1 / years)
    - 1
)

sharpe = (
    np.sqrt(len(trades)/years)
    * trades.mean()
    / trades.std()
)

max_dd = (
    equity / equity.cummax() - 1
).min()

profit_factor = (
    trades[trades > 0].sum()
    /
    abs(trades[trades < 0].sum())
)

print("="*60)
print("Trades         :", len(trades))
print("Total Return   :", f"{total_return:.2%}")
print("CAGR           :", f"{cagr:.2%}")
print("Sharpe         :", round(sharpe,2))
print("Max Drawdown   :", f"{max_dd:.2%}")
print("Profit Factor  :", round(profit_factor,2))
print("Average Trade  :", f"{trades.mean():.2%}")
print("="*60)

Trades         : 57
Total Return   : -7.31%
CAGR           : -0.75%
Sharpe         : -0.36
Max Drawdown   : -11.45%
Profit Factor  : 0.66
Average Trade  : -0.13%


# Conclusions

## Research Question

Can volatility compression combined with momentum improve forecasting performance?

## Evidence

- Compression identified periods of reduced market activity.
- Some breakouts occurred after compression.
- Performance remained inconsistent.
- Risk-adjusted returns were modest.

## Verdict

🔴 Rejected as a Standalone Strategy

The hypothesis contains some intuitive appeal, but the resulting signal lacked sufficient robustness.

Compression may be more useful as a filter than as a primary trading signal.